In [1]:
import torch
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import lightning.pytorch as pl
from lightning.pytorch.callbacks import RichProgressBar, Timer, LearningRateFinder
import omegaconf


# Add the prime_torch file to the system path so we can import it
import sys
sys.path.append("/glade/u/home/cobrien/prime/prime_lib/primesw")
from data import SWDataset, SWDataModule
from prime_torch import crps, SWRegressor

In [2]:
torch.set_float32_matmul_precision('medium')

config = '/glade/u/home/cobrien/prime/prime_lib/configs/plasmasheet.yaml'
cfg = omegaconf.OmegaConf.load(
    config
)

# Running .setup() with this DataModule will take like an hour. Find a way to cache this dataloader
datamodule = SWDataModule(
    target_features = cfg.data.target_features,
    input_features = cfg.data.input_features,
    position_features = cfg.data.position_features,
    interp_flags = cfg.data.interp_flags,
    region = cfg.data.region,
    cuts = cfg.data.cuts,
    cadence = cfg.data.cadence,
    interpolate = cfg.data.interpolate,
    window = cfg.data.window,
    stride = cfg.data.stride,
    interp_frac = cfg.data.interp_frac,
    trn_bounds = cfg.data.trn_bounds,
    val_bounds = cfg.data.val_bounds,
    tst_bounds = cfg.data.tst_bounds,
    batch_size = cfg.opt.batch_size,
    num_workers = cfg.opt.num_workers,
    datastore = cfg.data.datastore,
    in_key = cfg.data.in_key,
    tar_key = cfg.data.tar_key,
)

model = SWRegressor.load_from_checkpoint(
    "/glade/u/home/cobrien/data/prime/tensorboard_logs/pstesting/version_11/checkpoints/epoch=99-step=3600.ckpt"
)

2026-02-10 11:05:55.439 | INFO     | data:__init__:211 - Dataset cut density_despike_10


In [3]:
# Load the data
wind_data = pd.read_hdf('/glade/u/home/cobrien/data/magnetotail/resampled_datasets.h5', key = 'wind_full', mode = 'r')
themis_data = pd.read_csv('/glade/u/home/cobrien/data/themis/combined_loc_themis_cleaned.csv', index_col=None)
themis_data = themis_data.drop(index = [0,1])
themis_data['datetime'] = pd.to_datetime(themis_data['datetime'], utc = True)

In [4]:
# Make the position inputs and cut the input wind data to just the interval covered by the themis data
wind_data = wind_data.loc[
    (wind_data['Epoch'] <= themis_data['datetime'].max())&
    (wind_data['Epoch'] >= themis_data['datetime'].min() - pd.Timedelta(seconds = (model.stride + model.window) * 100)),
    :
]
positions = pd.DataFrame(wind_data['Epoch'], columns = ['Epoch'])
positions['Rx_int'] = np.interp(positions['Epoch'], themis_data['datetime'], themis_data['X_GSM'])
positions['Ry_int'] = np.interp(positions['Epoch'], themis_data['datetime'], themis_data['Y_GSM'])
positions['Rz_int'] = np.interp(positions['Epoch'], themis_data['datetime'], themis_data['Z_GSM'])

In [5]:
model.predict(wind_data, positions)

RuntimeError: The expanded size of the tensor (467) must match the existing size (667) at non-singleton dimension 0.  Target sizes: [467, -1].  Tensor sizes: [667, 30]

In [ ]:
in_scaled = wind_data.loc[:, model.in_norm.keys()].copy() # Get just the keys used for prediction
for feature in model.in_norm.keys(): # Scale each input feature DOWN
    in_scaled[feature] = (in_scaled[feature] - model.in_norm[feature][0])/model.in_norm[feature][1]

# Turn in_scaled into a numpy array of the correct shape
in_arr = np.zeros((len(positions) - model.window, model.window, len(model.in_norm.keys())))
for i, idx in enumerate(in_scaled.index):
    if i < model.window:
        continue
    in_arr[i - model.window, :, :] = in_scaled.loc[(idx - model.window - model.stride):(idx - model.stride - 1), :]

pos_scaled = position.iloc[model.window:].loc[:, model.pos_norm.keys()].copy() # Get just the position elements
for feature in model.pos_norm.keys(): # Scale each position DOWN
    pos_scaled[feature] = (pos_scaled[feature] - model.pos_norm[feature][0])/model.pos_norm[feature][1]

In [8]:
in_arr.shape

(467, 200, 14)

In [10]:
positions

,Epoch,Rx_int,Ry_int,Rz_int
9129797,2023-11-05 02:28:20+00:00,-8.987770,5.435874,-1.841249
9129798,2023-11-05 02:30:00+00:00,-8.987770,5.435874,-1.841249
9129799,2023-11-05 02:31:40+00:00,-8.987770,5.435874,-1.841249
9129800,2023-11-05 02:33:20+00:00,-8.987770,5.435874,-1.841249
9129801,2023-11-05 02:35:00+00:00,-8.987770,5.435874,-1.841249
...,...,...,...,...
9130459,2023-11-05 20:51:40+00:00,-11.434947,0.466296,-1.903945
9130460,2023-11-05 20:53:20+00:00,-11.425310,0.457339,-1.904387
9130461,2023-11-05 20:55:00+00:00,-11.405915,0.439427,-1.905274
9130462,2023-11-05 20:56:40+00:00,-11.386345,0.421481,-1.906161


In [ ]:
mms_data = pd.read_hdf("/glade/u/home/cobrien/data/combined_data.h5", key = "mms_1min_labeled_indexed")

In [26]:
mms_data[(mms_data['stable'] == 1)&(mms_data['modified_named_label'] == 'solar wind')].columns

Index(['Epoch', 'probe', 'ratio_max_width', 'ratio_high_low', 'norm_Btot',
       'small_energy_mean', 'large_energy_mean', 'temp_total', 'r_gse_x',
       'r_gse_y', 'r_gse_z', 'r_gsm_x', 'r_gsm_y', 'r_gsm_z', 'mlat', 'mlt',
       'raw_named_label', 'modified_named_label', 'transition_name',
       'mms1_dis_bulkv_gse_fast_0', 'mms1_dis_bulkv_gse_fast_1',
       'mms1_dis_bulkv_gse_fast_2', 'mms1_dis_numberdensity_fast', 'SW_table',
       'mms1_dis_energyspectr_omni_fast_0',
       'mms1_dis_energyspectr_omni_fast_1',
       'mms1_dis_energyspectr_omni_fast_2',
       'mms1_dis_energyspectr_omni_fast_3',
       'mms1_dis_energyspectr_omni_fast_4',
       'mms1_dis_energyspectr_omni_fast_5',
       'mms1_dis_energyspectr_omni_fast_6',
       'mms1_dis_energyspectr_omni_fast_7',
       'mms1_dis_energyspectr_omni_fast_8',
       'mms1_dis_energyspectr_omni_fast_9',
       'mms1_dis_energyspectr_omni_fast_10',
       'mms1_dis_energyspectr_omni_fast_11',
       'mms1_dis_energyspectr_o